# 7. Explaining a distance: its split over the critical points, and back to the robots

The taxonomy says *which* recordings behave alike. This notebook shows how the same method says *why* two
behaviours differ.

The sliced-Wasserstein distance of order one is a sum of terms, one per direction and per matched pair of points.
It therefore **splits exactly over the points of the two diagrams**: each term is shared equally between the two
elements it matches, a projection onto the diagonal standing for the point it projects. In a lower-star filtration,
every coordinate of a diagram point is the value of the field at one seed, a critical point (minimum, maximum or
saddle). Each point passes its share, half and half, to the seeds of its birth and its death, so that the whole
distance lies on critical points, each of which has a meaning in the motion (a sink or source of $\varphi$, a
rotation centre of $\psi$, a saddle between two basins).

With the seeds fixed, every step from the robot positions to a field value is differentiable:

positions $x_i(t)$ → velocities (forward differences) → field at the seeds (closed form).

Each seed's share is then divided among the robot positions in proportion to the size of the derivative of its
value with respect to each of them.

The comparison summary, the distribution of the field values (32 quantiles), splits in the same way, but its terms
land on whichever seeds happen to hold each quantile.

In [ ]:
import sys, pickle, subprocess
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "src"), str(ROOT)]
import numpy as np
import matplotlib.pyplot as plt
import figlib as L
L.style()
PAPER = ROOT / "results" / "epuck" / "paper"
load = lambda p: pickle.load(open(p, "rb"))

## Running the attribution

`analysis/attribution.py` re-implements the computation in PyTorch (float64) for the Top-1 exemplars of Directional
Gate and Foraging, on the vorticity potential $\psi_\sigma$ at $\sigma_3$ (the field that separates these two
missions most strongly). It uses the GPU when one is free (about two minutes in all, including the
stability and reproducibility tests). `analysis/attribution_pairs.py` runs it on the ten pairs of missions, each on
the potential that separates the pair better. Set `RUN = True` to recompute; otherwise the saved result is read.

In [ ]:
RUN = False
if RUN:
    subprocess.run([sys.executable, str(ROOT / "analysis" / "attribution.py")], check=True, cwd=ROOT)
A = load(PAPER / "attribution.pkl")
print(f"{A['missions'][0]} run {A['run_A'][1]}  vs  {A['missions'][1]} run {A['run_B'][1]};  field {A['field']} at sigma_{A['scale'] + 1}")

## 1. The differentiable chain is exactly the pipeline

Before any gradient, every step is checked against what the pipeline stored: the fields at the seeds (stored in
float32), the six diagrams, and the two distances.

In [ ]:
for k, v in A["check"].items():
    print(f"{k:40s} {v}")

Fields agree to the float32 storage precision ($\sim10^{-7}$), diagrams and distances to machine precision,
the pairing found on the recomputed float64 field is the same as on the stored one, and the split sums to the
distance.

## 2. Where the distance lies

For each summary, the share of the distance carried by each seed of the D-Gate run. Each seed is classified from
its link in the alpha complex: minimum (empty lower link), maximum (empty upper link), saddle (lower or upper link
in two pieces or more), otherwise ordinary.

In [ ]:
for summary in ("persistence", "values"):
    sh = A[f"{summary}_shares_A"]
    print(f"{summary:12s} distance {A[f'{summary}_distance']:.5f}: {sh['n']:3d} of {len(A[f'{summary}_share_A'])} seeds carry it, "
          f"{100 * sh['critical']:5.1f} % of it on critical points, {100 * sh['extrema']:5.1f} % on extrema")
print(f"gradient of the distance on critical points: {100 * A['persistence_grad_shares_A']['critical']:.1f} %")
names = ["Ord0", "Ord1", "Rel1", "Rel2", "Ext0", "Ext1"]
print("\nlargest diagram points of the D-Gate run (persistence diagrams, share of the distance):")
for p in A["points_A"][:5]:
    print(f"  {names[p['diagram']]:4s} ({p['birth']:+.4f}, {p['death']:+.4f})  {p['birth_type']:>6s} -> {p['death_type']:<6s}"
          f"  share {p['share'] / A['persistence_distance']:.3f}")

In [ ]:
import make_figures as M
from IPython.display import Image, display
M.fig_attribution()
display(Image(filename=str(ROOT / "figures/fig_attribution.png"), width=950))

**What it shows.**
* **(a)** The topological split sits on critical points only. The two rotation centres of the traffic, a maximum and
  a minimum of $\psi$, carry the largest shares; the rest spreads over the other critical points of the field,
  saddles included. The distance rests on the whole organisation of the field rather than on one structure.
* **(d)** The value split is spread over ordinary points that only happen to hold a quantile.
* **(b, e)** Carried to the position of every robot at every instant and accumulated where the robots were, both
  splits gather where the robots crowd and turn at the end of the corridor: these turns set the mean speed that
  normalises the potentials, on which every value of the field depends. The first and last instants are left out:
  with forward differences, their positions are not compensated by a neighbouring instant.
* **(c, f)** Averaged over the 50 runs of the D-Gate controller, each against the same Foraging exemplar, the
  topological split concentrates on the two rotation centres. The value split covers the whole positive cell.

## 3. Stability and reproducibility

In [ ]:
print("noise on positions | field change | split kept on the same seeds (persistence / values)")
for eps, r in A["stability"].items():
    print(f"   {1000 * eps:4.1f} mm          |  {100 * r['field_change']:5.1f} %    |   {r['persistence_same']:.2f} / {r['values_same']:.2f}")
rp = A["reproducibility"]; ce = rp["centres"]
print(f"\n{len(rp['runs'])} runs of the controller: the two extrema with the largest topological shares are a max and a min "
      f"in {ce['n_max_min']} runs (median share {ce['share_median']:.2f}); rms displacement of the centres "
      f"{100 * np.mean(list(ce['spread_m'].values())):.0f} cm")
print(f"half of the mean map lies in {100 * rp['persistence']['area_half']:.0f} % of the arena (persistence) "
      f"vs {100 * rp['values']['area_half']:.0f} % (values)")
print(f"mean overlap of two single maps: persistence {rp['persistence']['mean_overlap']:.2f}, values {rp['values']['mean_overlap']:.2f}")

Both splits are stable under small noise on the positions: the table above gives, for each noise level, the share
of each split that stays on the same seeds, and the topological split keeps more of it at every noise tested.
Across runs, the topological split keeps its **structure**: a maximum and a minimum of $\psi$ carry its largest
shares. But the centres move by tens of centimetres from run to run, so two single topological maps overlap less
than two diffuse value maps. What recurs is the structure, not its exact position.

**Limits.** The split of the distance is exact; its passage to the robots is first order. With the seeds and the
pairing held fixed, it says which structures and which robots carry a distance, not how far the trajectories
would have to change before the pairing itself changes. Here it is shown for one pair of missions, one field and
one scale; `analysis/attribution_pairs.py` repeats it on the ten pairs.